#  **Практическое занятие №6. Компьютерное зрение. Свёрточные нейронные сети.**

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## CIFAR 10

In [ ]:
from torchvision.datasets import CIFAR10
from torchvision.transforms.v2 import Compose, PILToTensor, ToDtype, Normalize

cifar_transform = Compose([
    PILToTensor(),
    ToDtype(torch.float32, scale=True),
    Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25))
])

cifar = CIFAR10(root='cifar', download=True, train=True, transform=cifar_transform)

cifar_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,4))

for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(cifar[i][0].permute(1, 2, 0) * 0.25 + 0.5)
    plt.xticks([])
    plt.yticks([])
    plt.title(cifar_names[cifar[i][1]])

plt.show()

In [ ]:
from torch.utils.data import random_split

train_set, valid_set = random_split(cifar, (0.95, 0.05))

## Simple network

In [ ]:
pip install torchsummary

In [ ]:
# build model

from torch import nn
from torchsummary import summary

model = nn.Sequential(
    nn.Flatten(start_dim=-3),
    nn.Linear(in_features=3072, out_features=512),
    nn.ReLU(),
    nn.Linear(in_features=512, out_features=64),
    nn.ReLU(),
    nn.Linear(in_features=64, out_features=10),
    nn.Sigmoid()
).to(device)

summary(model, (3, 32, 32))

In [ ]:
# prepare

from torch.utils.data import DataLoader

optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

train_loader = DataLoader(train_set, batch_size=8)
valid_loader = DataLoader(valid_set, batch_size=8)

In [ ]:
# train model

Можно снизить количество бойлерплейта и сделать код более структурированным за счет использования дополнительных библиотек

## PyTorch Ignite

See https://pytorch-ignite.ai

See https://pytorch-ignite.ai/tutorials/beginner/01-getting-started

In [ ]:
pip install pytorch-ignite

In [ ]:
# create trainer and evaluators

from ignite.engine import create_supervised_trainer, create_supervised_evaluator
from ignite.metrics import Accuracy, Loss

trainer = create_supervised_trainer(model, optimizer, criterion, device)

metrics = {
    "accuracy": Accuracy(),
    "loss": Loss(criterion)
}

train_evaluator = create_supervised_evaluator(model, metrics=metrics, device=device)
valid_evaluator = create_supervised_evaluator(model, metrics=metrics, device=device)

In [ ]:
# add logging

from ignite.engine import Events

def log_iter_loss(engine):
    print(f"Epoch[{engine.state.epoch}] - Iter[{engine.state.iteration}]: loss = {engine.state.output}")

trainer.add_event_handler(Events.ITERATION_COMPLETED(every=1000), log_iter_loss)

def compute_epoch_results(engine):
    train_evaluator.run(train_loader)
    valid_evaluator.run(valid_loader)

trainer.add_event_handler(Events.EPOCH_COMPLETED, compute_epoch_results)

def log_epoch_results(engine, label=""):
    result = ', '.join([f"{m} = {v}" for m, v in engine.state.metrics.items()])
    print(f"{label} Res:", result)

train_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_epoch_results, label="Train")
valid_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_epoch_results, label="Valid")

In [ ]:
# add lr scheduler

from ignite.handlers.param_scheduler import ReduceLROnPlateauScheduler

scheduler = ReduceLROnPlateauScheduler(
    optimizer,
    metric_name="loss",
    factor=0.5,
    patience=1,
    threshold=0.05
)

def print_lr():
    for param_group in optimizer.param_groups:
        print(f"Optimizer learning rate = {param_group['lr']}")

valid_evaluator.add_event_handler(Events.COMPLETED, scheduler)
valid_evaluator.add_event_handler(Events.COMPLETED, print_lr)

In [ ]:
# run training loop

trainer.run(train_loader, 2)

In [ ]:
model.eval()

plt.figure(figsize=(15,4))

for i in range(20):
    x, y_true = valid_set[i]
    y_pred = torch.argmax(model(x.to(device)))
    plt.subplot(2, 10, i + 1)
    plt.imshow(x.permute(1, 2, 0) * 0.25 + 0.5)
    plt.xticks([])
    plt.yticks([])
    plt.title(f'True = {cifar_names[y_true]}\nPred = {cifar_names[y_pred]}')

plt.show()

## Convolutional network

See https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html#torch.nn.Conv2d

See https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html#torch.nn.MaxPool2d

In [ ]:
model = nn.Sequential(

    nn.Conv2d(3, 16, 5),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),   # out shape ?, params count = ?

    nn.Conv2d(16, 64, 5),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),   # out shape ?, params count = ?

    nn.Conv2d(64, 256, 5),
    nn.ReLU(),            # out shape ?, params count = ?

    nn.Flatten(-3),
    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Linear(64, 10)

).to(device)

summary(model, (3, 32, 32))

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
criterion = torch.nn.CrossEntropyLoss()

train_loader = DataLoader(train_set, batch_size=8)
valid_loader = DataLoader(valid_set, batch_size=8)

trainer = create_supervised_trainer(model, optimizer, criterion, device)

metrics = {
    "accuracy": Accuracy(),
    "loss": Loss(criterion)
}

train_evaluator = create_supervised_evaluator(model, metrics=metrics, device=device)
valid_evaluator = create_supervised_evaluator(model, metrics=metrics, device=device)

trainer.add_event_handler(Events.ITERATION_COMPLETED(every=1000), log_iter_loss)
trainer.add_event_handler(Events.EPOCH_COMPLETED, compute_epoch_results)
train_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_epoch_results, label="Train")
valid_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_epoch_results, label="Valid")

In [ ]:
trainer.run(train_loader, 2)

In [ ]:
model.eval()

plt.figure(figsize=(15,4))

for i in range(20):
    x, y_true = valid_set[i]
    y_pred = torch.argmax(model(x.to(device)))
    plt.subplot(2, 10, i + 1)
    plt.imshow(x.permute(1, 2, 0) * 0.25 + 0.5)
    plt.xticks([])
    plt.yticks([])
    plt.title(f'True = {cifar_names[y_true]}\nPred = {cifar_names[y_pred]}')

plt.show()

## Additionals

### Data Augmentation

See https://pytorch.org/vision/main/transforms.html

In [ ]:
from torchvision.transforms.v2 import RandomHorizontalFlip

cifar_transform_flip = Compose([
    RandomHorizontalFlip(1.0),
    PILToTensor(),
    ToDtype(torch.float32, scale=True),
    Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25))
])

cifar_flip = CIFAR10(root='cifar', download=True, train=True, transform=cifar_transform_flip)

In [ ]:
plt.figure(figsize=(15,2))
plt.suptitle("Original")

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(cifar[i][0].permute(1, 2, 0) * 0.25 + 0.5)
    plt.xticks([])
    plt.yticks([])


plt.figure(figsize=(15,2))
plt.suptitle("Flipped")

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(cifar_flip[i][0].permute(1, 2, 0) * 0.25 + 0.5)
    plt.xticks([])
    plt.yticks([])


plt.show()

### Ignite utilities

Adding checkpoints - https://pytorch.org/ignite/generated/ignite.handlers.checkpoint.ModelCheckpoint.html#modelcheckpoint

Pretty loggers - https://pytorch-ignite.ai/how-to-guides/10-loggers